In [7]:
# Add autoreload at the top of the notebook you're working on 
# in order for it to auto refresh when you change the 'project_package'
%load_ext autoreload
%autoreload 2

# Initialize dummy dataset

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

import plotly.express as px
import pandas as pd
import numpy as np
from langchain_chroma import Chroma
from flashrank import Ranker

from project_package.data_preprocessing.utils import create_user_preference
from project_package.modeling.recommendation_utils import (
    construct_rec_train_dataset,get_embedding_model,
    VectorstoreLoader,doc_template_fill_in,keep_n_labels,
    recommendation_doc_id_pipeline
    )
from project_package.visualization import sankey_plot
from project_package.data_preprocessing.default import ITEM_ML
load_dotenv()  # load env variables from .evn
root_directory = Path(os.getcwd()).parent.parent  #NOTE: update of notebook location changed

g:\Python\envs\capstone_test3\Lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


## 1. Load data

Load item-content metadata

In [9]:
movie_metadata = pd.read_csv(
    root_directory / 'data/external/movielense_25m/movies_metadata.csv',
    usecols = ['id','original_title','overview','genres','original_language',
               'adult','runtime','revenue','vote_average','vote_count']
    )
movie_metadata['genres'] = np.array(movie_metadata['genres'].apply(lambda x:[item['name'] for item in eval(x)]))

movie_metadata['id'] = pd.to_numeric(movie_metadata['id'], errors='coerce')
movie_metadata.dropna(subset=['id'],inplace=True)
movie_metadata['id'] = movie_metadata['id'].astype(int)
movie_metadata.rename(columns={'id':'movieId'},inplace=True)

movie_metadata = movie_metadata.drop_duplicates('movieId').reset_index(drop=True)

movie_metadata['genres'] = movie_metadata['genres'].apply(
    lambda x: ['Unknown'] if isinstance(x, list) and len(x) == 0 else x
)

map_dict = {"False":"People of all age","True":"Adult only"}
movie_metadata['adult'] = movie_metadata['adult'].map(map_dict)
movie_metadata.head(3)

,adult,genres,movieId,original_language,original_title,overview,revenue,runtime,vote_average,vote_count
0,People of all age,"[Animation, Comedy, Family]",862,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",373554033.0,81.0,7.7,5415.0
1,People of all age,"[Adventure, Fantasy, Family]",8844,en,Jumanji,When siblings Judy and Peter discover an encha...,262797249.0,104.0,6.9,2413.0
2,People of all age,"[Romance, Comedy]",15602,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,0.0,101.0,6.5,92.0


Load user rating data

In [10]:
# should sort reviews by user first
user_reviews = pd.read_csv(
    root_directory / 'data/external/movielense_25m/ratings_small.csv',
    usecols = ['userId','movieId','rating'],
    )
user_reviews.head(5)

,userId,movieId,rating
0,1,31,2.5
1,1,1029,3.0
2,1,1061,3.0
3,1,1129,2.0
4,1,1172,4.0


Retrieve user preference data

In [11]:
# Generate user preference data

if not os.path.exists(root_directory / 'data/external/movielense_25m/user_preferences.csv'):
    criteria_dict = dict(
        genres = (0.3,'multiple'),
        original_language = (0.1,'single'),
        adult = (0.4,'single')
    )

    preference_df = create_user_preference(
        movie_metadata,'movieId',
        user_reviews,'userId',
        criteria_dict,
        'rating',
        user_batch=200
    )
    preference_df.to_csv(root_directory / 'data/external/movielense_25m/user_preferences.csv',index =False)
else:
    preference_df = pd.read_csv(root_directory / 'data/external/movielense_25m/user_preferences.csv')

preference_df.head(3)

,userId,genres,original_language,adult
0,1,['Comedy'],['en'],['People of all age']
1,2,['Drama'],['en'],['People of all age']
2,3,['Drama'],"['en', 'fr']",['People of all age']


In [12]:
doc_template = """
The movie title:
{}

The movie overview:
{}

The genres:
{}

The movie is for:
{}
"""

user_profile_template = """
Favorite genres:
{}
Favorite languages:
{}
Preferred movie PG type:
{}
"""

embedding_model = get_embedding_model(
    huggingface_model_path="BAAI/bge-small-en-v1.5",  # NOTE: Change this embedding to foodbert later
    local_model_name="bge-small",
    device="cuda"
)
chroma_path = root_directory / "data/external/movielense_25m/chroma_db"  #NOTE: Change the Path later

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Get the vectorstores

In [13]:
#NOTE: Run the cell again when a batch is failed to continue

store_document = False  # Set to True when you need to run the embedding loading again
vector_loader = None

if store_document:
    if vector_loader is None:
        vector_loader = VectorstoreLoader(
            collection_name="movielens_collection",  #NOTE: Change the collection name later,
            embedding = embedding_model,
            doc_template = doc_template,
            input_data = movie_metadata,
            format_cols = ['original_title','overview','genres','adult'],
            meta_cols = ['genres','original_language','adult','runtime','vote_average','vote_count','movieId'],  # include item ID for later filter tasks
            persist_directory = chroma_path,
            docID_col = 'movieId'  #NOTE: Change the ID later
        )

    load_finished =  vector_loader.add_initial_documents(batch_size=1000)
    if load_finished:
        vectorstore = vector_loader.return_vectorstore()
        del vector_loader  # to reduce memory usage
else:
    vectorstore = Chroma(
        collection_name="movielens_collection",
        embedding_function=embedding_model,
        persist_directory=chroma_path
    )

Create a collection for user preferences

In [14]:
#NOTE: Run the cell again when a batch is failed to continue

store_user_pref = False  # Set to True when you need to run the embedding loading again
vector_loader = None

if store_user_pref:
    if vector_loader is None:
        vector_loader = VectorstoreLoader(
            collection_name="movielens_user_preference",  #NOTE: Change the collection name later,
            embedding = embedding_model,
            doc_template = user_profile_template,
            input_data = preference_df,
            format_cols = ['genres','original_language','adult'],
            meta_cols = None,
            persist_directory = chroma_path,
            docID_col = 'userId'  #NOTE: Change the ID later
        )

    load_finished =  vector_loader.add_initial_documents(batch_size=1000)
    if load_finished:
        user_vectorstore = vector_loader.return_vectorstore()
        del vector_loader  # to reduce memory usage
else:
    user_vectorstore = Chroma(
        collection_name="movielens_user_preference",
        embedding_function=embedding_model,
        persist_directory=chroma_path
    )

## 2. Load trained recommendation models and model dataset format

In [15]:
k = 10  # number of recommendation to create

# load data to Rectools format
dataset = construct_rec_train_dataset(
    user_reviews,
    movie_metadata,
    preference_df,
    use_test_cols=True
)

In [16]:
#NOTE: Load the models, for real dataset, we might need to load this from S3 or google drive
svd_model = load_model(root_directory / "models/movielens_test/svd_recommendation_model.pkl")
als_model = load_model(root_directory / "models/movielens_test/als_recommendation_model.pkl")
lightfm_model = load_model(root_directory / "models/movielens_test/lightFM_recommendation_model.pkl")

## 3. The change in label distribution through recommendation pipeline

In [11]:
# target = ['Horror', 'Thriller']

# df_filtered = movie_metadata[
#     # movie_metadata["genres"].map(lambda x: target.issubset(set(x)))
#     movie_metadata["genres"].map(lambda x: x == target)
# ]
# df_filtered

First we will create some random queries to check the recommendation pipeline output

In [17]:
# a random sample of items to be used as queries
sample_df = movie_metadata.sample(20,random_state=0)
sample_documents = doc_template_fill_in(doc_template,sample_df,['original_title','overview','genres','adult'],docID_col=ITEM_ML)
sample_documents = [doc.page_content for doc in sample_documents]

reranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",  # NOTE: Can change to a different Flashrank model of your liking
    cache_dir=os.environ["FLASHRANK_PATH"]
)

First, we look at the label distribution of a category before any filtering step

In [18]:
fig = px.histogram(
    keep_n_labels(movie_metadata,'genres',True,20), 
    x="genres",
    log_y=True,
    height=400,
    title='Top 20 label counts in full dataset'
    )
fig.update_xaxes(categoryorder='total descending').show()

Now we look at the dsitrbution of the feature labels after each recommendation step. We are using multiple different queries to give a more statistical significant view.

In [19]:
plot_df = pd.DataFrame()

for query in sample_documents:
    buffer_df, _ = recommendation_doc_id_pipeline(
        movie_metadata,
        vectorstore,
        reranker,
        query,
        embedding_model=embedding_model,
        toy_dataset=True,
        features = ['genres']
    )
    plot_df = pd.concat((plot_df,buffer_df))

plot_df = plot_df.reset_index(drop=True)

In [20]:
fig = px.histogram(
    keep_n_labels(plot_df,'genres',True,20), 
    x="genres",
    color='pipeline_step', barmode='group',log_y=True,
    height=600,
    width=1000,
    title='Top 20 label after recommendation pipeline')
fig.update_xaxes(categoryorder='total descending').show()

We try to remove certain biases with weight from the queries and see how it performs

In [21]:
no_bias_df = pd.DataFrame()

for query in sample_documents:
    buffer_df, _ = recommendation_doc_id_pipeline(
        movie_metadata,
        vectorstore,
        reranker,
        query,
        remove_biases = [("Comedy and Drama",1.5)],
        embedding_model=embedding_model,
        toy_dataset=True,
        features = ['genres'],
        neg_rank_bias="Comedy and Drama"
    )
    no_bias_df = pd.concat((no_bias_df,buffer_df))

no_bias_df = no_bias_df.reset_index(drop=True)

In [22]:
fig = px.histogram(
    keep_n_labels(no_bias_df,'genres',True,20), 
    x="genres",
    color='pipeline_step', barmode='group',log_y=True,
    height=600,
    width=1000,
    title='Top 20 label after recommendation pipeline with removing bias')
fig.update_xaxes(categoryorder='total descending').show()

Now, for another case, we try to add some bias to the query to shift the recommendations

In [23]:
add_bias_df = pd.DataFrame()

for query in sample_documents:
    buffer_df, _ = recommendation_doc_id_pipeline(
        movie_metadata,
        vectorstore,
        reranker,
        query,
        add_biases = [("Horror, Animation and War",1.5)],
        embedding_model=embedding_model,
        toy_dataset=True,
        features = ['genres'],
        pos_rank_bias = "Horror, Animation and War"
    )
    add_bias_df = pd.concat((add_bias_df,buffer_df))

add_bias_df = add_bias_df.reset_index(drop=True)

In [24]:
fig = px.histogram(
    keep_n_labels(add_bias_df,'genres',True,20), 
    x="genres",
    color='pipeline_step', barmode='group',log_y=True,
    height=600,
    width=1000,
    title='Top 20 label after recommendation pipeline with added bias')
fig.update_xaxes(categoryorder='total descending').show()

# 4. Model's recommendation evaluation with sankey plot

We will use a custom query to check how a category labels is being updated through recommendation pipeline

In [25]:
# a single test query
sample_item = movie_metadata.loc[movie_metadata.movieId==34574]
test_query = doc_template.format(*sample_item[['original_title','overview','genres','adult']].to_numpy()[0])

In [26]:
test_user_profile = user_profile_template.format(
    ['Comedy','War'],
    ['en', 'fr'],
    ['People of all age']
)

We are testing a single query for a BM25 recommendation pipeline with default setting, removing bias, and adding bias

In [27]:
# Single query pipeline test
test_query_df,_ = recommendation_doc_id_pipeline(
    movie_metadata,
    vectorstore,
    reranker,
    test_query,
    embedding_model=embedding_model,
    toy_dataset=True,
    features = ['genres'],
    include_fulldata = True
)

test_query_df = keep_n_labels(test_query_df,'genres',True,10)

# Plot the sankey plot
sankey_plot(test_query_df,'genres',"Sankey plot for category labels through pipeline",True,['Comedy','Drama'])

Same as previous section, we will try to remove some bias from the semantic search

In [28]:
# Single query pipeline test
no_bias_df,_ = recommendation_doc_id_pipeline(
    movie_metadata,
    vectorstore,
    reranker,
    test_query,
    embedding_model=embedding_model,
    remove_biases = [("Comedy and Drama",1.5)],
    toy_dataset=True,
    features = ['genres'],
    neg_rank_bias = "Comedy and Drama",
    include_fulldata = True
)
no_bias_df = keep_n_labels(no_bias_df,'genres',True,10)

# Plot the sankey plot
sankey_plot(no_bias_df,'genres',"Sankey plot for category labels through pipeline",True,['Comedy','Drama'])

Then again, we check the difference in the sankey plot when we add some bias to the query

In [29]:
# Single query pipeline test
add_bias_df,_ = recommendation_doc_id_pipeline(
    movie_metadata,
    vectorstore,
    reranker,
    test_query,
    embedding_model=embedding_model,
    add_biases = [("Horror, Animation and War",1.5)],
    toy_dataset=True,
    features = ['genres'],
    pos_rank_bias = "Horror, Animation and War",
    include_fulldata = True
)
add_bias_df = keep_n_labels(add_bias_df,'genres',True,10)

# Plot the sankey plot
sankey_plot(add_bias_df,'genres',"Sankey plot for category labels through pipeline",True,['Horror','Animation','War'])

We can also test other recommendation model to see how it performs

**SVD model**

In [33]:
test_query_df,test_rank = recommendation_doc_id_pipeline(
    movie_metadata,
    vectorstore,
    reranker,
    test_query,
    dataset=dataset,
    embedding_model=embedding_model,
    recommendation_model=svd_model,
    user_vectorstore=user_vectorstore,
    user_profile=test_user_profile,
    model_type = 'collab',
    toy_dataset=True,
    features = ['genres'],
    include_fulldata = True,
    random_state = 0
)

test_query_df = keep_n_labels(test_query_df,'genres',True,10)

# Plot the sankey plot
sankey_plot(test_query_df,'genres',"Sankey plot for SVD model",True,['Comedy','Drama'])

**ALS model**

In [36]:
test_query_df,test_rank = recommendation_doc_id_pipeline(
    movie_metadata,
    vectorstore,
    reranker,
    test_query,
    dataset=dataset,
    embedding_model=embedding_model,
    recommendation_model=als_model,
    user_vectorstore=user_vectorstore,
    user_profile=test_user_profile,
    model_type = 'collab',
    toy_dataset=True,
    features = ['genres'],
    include_fulldata = True,
    random_state = 0
)

test_query_df = keep_n_labels(test_query_df,'genres',True,10)

# Plot the sankey plot
sankey_plot(test_query_df,'genres',"Sankey plot for ALS model",True,['Comedy','Drama'])

**LightFM hybrid model**

In [38]:
test_query_df,test_rank = recommendation_doc_id_pipeline(
    movie_metadata,
    vectorstore,
    reranker,
    test_query,
    dataset=dataset,
    embedding_model=embedding_model,
    recommendation_model=lightfm_model,
    user_vectorstore=user_vectorstore,
    user_profile=test_user_profile,
    model_type = 'hybrid',
    toy_dataset=True,
    features = ['genres'],
    include_fulldata = True,
    random_state = 0
)

test_query_df = keep_n_labels(test_query_df,'genres',True,10)

# Plot the sankey plot
sankey_plot(test_query_df,'genres',"Sankey plot for LightFM model",True,['Comedy','Drama'])